In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/ishantjuyal/emotions-in-text/Emotion_final.csv


In [2]:
df = pd.read_csv('/kaggle/input/datasets/ishantjuyal/emotions-in-text/Emotion_final.csv')

# View the first 5 rows
df.head()

,Text,Emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [3]:
text = df['Text']

In [4]:
text[:100]

0                               i didnt feel humiliated
1     i can go from feeling so hopeless to so damned...
2      im grabbing a minute to post i feel greedy wrong
3     i am ever feeling nostalgic about the fireplac...
4                                  i am feeling grouchy
                            ...                        
95    i feel like throwing away the shitty piece of ...
96    im starting to feel wryly amused at the banal ...
97    i find every body beautiful and only want peop...
98    i hear are owners who feel victimized by their...
99    i say goodbye to the fam theyre all sad a cryi...
Name: Text, Length: 100, dtype: object

In [5]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

# 1. Initialize a BPE tokenizer model
tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))

# 2. Add a pre-tokenizer (e.g., split on whitespace/punctuation)
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

# 3. Define the trainer with target vocabulary size and special tokens
trainer = trainers.BpeTrainer(
    vocab_size=5000, 
    special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]
)

# 4. Train the tokenizer directly on your text Series
tokenizer.train_from_iterator(text, trainer)

In [6]:
# 5. Test encoding a sentence
output = tokenizer.encode("os akhia hana doli charan")
print("Tokens:", output.tokens)
print("Token IDs:", output.ids)

Tokens: ['os', 'ak', 'hi', 'a', 'h', 'ana', 'do', 'li', 'char', 'an']
Token IDs: [2036, 622, 665, 55, 62, 4459, 156, 112, 821, 84]


In [7]:
import tiktoken

# Load the BPE encoding used by GPT-4 / GPT-3.5
enc = tiktoken.get_encoding("cl100k_base")

# Encode a sample sentence into token IDs
encoded_ids = enc.encode("os akhia hana doli charan")
print("Token IDs:", encoded_ids)

# Decode back to text
decoded_text = enc.decode(encoded_ids)
print("Decoded:", decoded_text)

Token IDs: [437, 17774, 71, 689, 305, 3444, 294, 14559, 1181, 276]
Decoded: os akhia hana doli charan


## SentencePiece

In [8]:
pip install sentencepiece

Note: you may need to restart the kernel to use updated packages.


In [9]:
import sentencepiece as spm

# 1. Save the pandas Series to a text file line-by-line
text.to_csv('input.txt', index=False, header=False)

# 2. Train SentencePiece on the file
spm.SentencePieceTrainer.train(
    input='input.txt',
    model_prefix='bpe',
    vocab_size=5000,
    model_type='bpe'
)

# 3. Load and test the trained model
sp = spm.SentencePieceProcessor()
sp.load('bpe.model')

tokens = sp.encode_as_pieces("i am feeling nostalgic about the fireplace")
ids = sp.encode_as_ids("i am feeling nostalgic about the fireplace")

print("Tokens:", tokens)
print("Token IDs:", ids)

Tokens: ['▁i', '▁am', '▁feeling', '▁nostalgic', '▁about', '▁the', '▁fire', 'place']
Token IDs: [3, 87, 48, 1876, 115, 22, 3563, 4744]


sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: input.txt
  input_format: 
  model_prefix: bpe
  model_type: BPE
  vocab_size: 5000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_privacy: 0
  differential_priv

## WordPiece

In [10]:
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import WordPieceTrainer

# Create WordPiece tokenizer
tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))

# Pre-tokenization
tokenizer.pre_tokenizer = Whitespace()

# Configure trainer
trainer = WordPieceTrainer(
    vocab_size=30000,
    special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"]
)

text.to_csv("input.txt", index=False, header=False)

# Train on corpus
files = ["input.txt"]
tokenizer.train(files, trainer)

# Save tokenizer
tokenizer.save("wordpiece.json")

In [11]:
# Encode text
output = tokenizer.encode("Machine learning is amazing.")

print("Tokens:", output.tokens)
print("IDs:", output.ids)

# Decode back
print("Decoded:", tokenizer.decode(output.ids))

Tokens: ['Mach', '##ine', 'learning', 'is', 'amazing', '.']
IDs: [15438, 442, 2183, 212, 944, 14]
Decoded: Mach ##ine learning is amazing .
